# Capítulo 8 — Taller integrador: Python aplicado a Data Science

**Serie Ciencia de Datos · Tomo 1 — Introducción al Data Science**  
José Fernando Meza

Código del capítulo, extraído del libro. Repositorio: https://github.com/josefer76/serie-ciencia-de-datos

> Ejecute las celdas en orden: algunas dependen de variables definidas antes.
> Las celdas que usan `input()` esperan que usted escriba un valor.

## Soluciones de los ejercicios

### [Ex-L1-C8-01] — Pipeline de análisis de frecuencia de texto

In [ ]:
"""Pipeline de análisis de frecuencia de palabras, con cada etapa en su
función."""
 
SIGNOS_PUNTUACION = ".,;!?"
TOP_N = 10
 
 
def limpiar_texto(documento):
    """Devuelve el documento en minúsculas y sin signos de puntuación
    básicos."""
    texto_limpio = documento.lower()
    # Eliminamos cada signo de puntuación reemplazándolo por vacío.
    for signo in SIGNOS_PUNTUACION:
        texto_limpio = texto_limpio.replace(signo, "")
    return texto_limpio
 
 
def tokenizar(texto_limpio):
    """Divide un texto limpio en una lista de palabras (tokens)."""
    # .split() sin argumentos divide por cualquier cantidad de espacios.
    return texto_limpio.split()
 
 
def construir_vocabulario(corpus):
    """Construye el set de palabras únicas de todo el corpus."""
    vocabulario = set()
    for documento in corpus:
        tokens = tokenizar(limpiar_texto(documento))
        # .update() agrega todos los tokens; los duplicados se descartan solos.
        vocabulario.update(tokens)
    return vocabulario
 
 
def contar_frecuencias(corpus):
    """Cuenta las apariciones totales de cada palabra en el corpus."""
    frecuencias = {}
    for documento in corpus:
        tokens = tokenizar(limpiar_texto(documento))
        for token in tokens:
            # Patrón de conteo canónico: get(clave, 0) + 1.
            frecuencias[token] = frecuencias.get(token, 0) + 1
    return frecuencias
 
 
def top_palabras(frecuencias, cantidad):
    """Devuelve las 'cantidad' palabras más frecuentes, de mayor a menor."""
    # sorted sobre los items (tuplas). key=lambda toma la frecuencia (índice
    # 1).
    # reverse=True ordena de mayor a menor. [:cantidad] recorta al top.
    ranking = sorted(
        frecuencias.items(),
        key=lambda par: par[1],
        reverse=True,
    )
    return ranking[:cantidad]
 
 
# --- Orquestación del pipeline sobre un corpus de ejemplo ---
corpus = [
    "El analisis de datos guia las decisiones de negocio.",
    "Los datos son el nuevo petroleo de la economia.",
    "Sin datos limpios, el analisis de datos falla.",
    "La ciencia de datos transforma datos en valor.",
    "Decisiones basadas en datos superan a la intuicion.",
]
 
vocabulario = construir_vocabulario(corpus)
frecuencias = contar_frecuencias(corpus)
ranking = top_palabras(frecuencias, TOP_N)
 
# --- Informe ---
total_palabras = sum(frecuencias.values())
print(f"Tamaño del vocabulario (palabras únicas): {len(vocabulario)}")
print(f"Total de palabras en el corpus: {total_palabras}")
print(f"\nTop {TOP_N} palabras más frecuentes:")
for posicion, (palabra, frecuencia) in enumerate(ranking, start=1):
    # Desempaquetamos cada tupla (palabra, frecuencia) en el for.
    print(f"  {posicion:2}. '{palabra}': {frecuencia}")

### [Ex-L1-C8-02] — Similitud de idiomas II: la matriz de similitud completa

In [ ]:
"""Similitud de idiomas II: matriz de Jaccard generada con comprehensions."""
 
 
def indice_jaccard(set_a, set_b):
    """Calcula el Índice de Jaccard entre dos conjuntos.
 
    J(A, B) = |A ∩ B| / |A ∪ B|
 
    Args:
        set_a (set): primer conjunto de palabras.
        set_b (set): segundo conjunto de palabras.
 
    Returns:
        float: similitud de Jaccard en [0, 1].
    """
    try:
        interseccion = len(set_a & set_b)  # & -> elementos comunes
        union = len(set_a | set_b)         # | -> todos los elementos
        return interseccion / union
    except ZeroDivisionError:
        # Caso límite: ambos conjuntos vacíos -> unión vacía.
        # Convención de ESTE ejercicio: dos conjuntos vacíos se consideran
        # idénticos (1.0).
        return 1.0
 
 
# --- Vocabularios como diccionario de sets (clave: idioma; valor: set) ---
idiomas = {
    "Español":    {"casa", "sol", "gato", "flor", "agua",
                   "mano", "libro", "verde", "mar", "luz"},
    "Inglés":     {"house", "sun", "cat", "flower", "water",
                   "hand", "book", "green", "sea", "light"},
    "Portugués":  {"casa", "sol", "gato", "flor", "agua",
                   "mao", "livro", "verde", "mar", "luz"},
    "Francés":    {"maison", "soleil", "chat", "fleur", "eau",
                   "main", "livre", "vert", "mer", "lumiere"},
}
 
# Lista de nombres, para fijar un orden estable de filas y columnas.
nombres_idiomas = list(idiomas.keys())
 
# --- Construcción de la matriz con comprehensions ANIDADAS ---
# Para cada idioma de fila, Jaccard contra cada idioma de columna.
matriz_similitud = [
    [
        indice_jaccard(idiomas[idioma_fila], idiomas[idioma_columna])
        for idioma_columna in nombres_idiomas  # bucle interno: columnas
    ]
    for idioma_fila in nombres_idiomas          # bucle externo: filas
]
 
# --- Impresión de la matriz en formato tabular alineado ---
# :>10 alinea a la derecha en 10 espacios; :<12 a la izquierda.
encabezado = " " * 12 + "".join(f"{nombre:>10}" for nombre in nombres_idiomas)
print(encabezado)
 
for indice_fila, idioma_fila in enumerate(nombres_idiomas):
    fila_texto = f"{idioma_fila:<12}"
    for indice_columna in range(len(nombres_idiomas)):
        valor = matriz_similitud[indice_fila][indice_columna]
        fila_texto += f"{valor:>10.2f}"
    print(fila_texto)
 
# --- Análisis: par más y menos similar (ignorando la diagonal) ---
mayor_similitud = -1.0
menor_similitud = 2.0
par_mas_similar = ()
par_menos_similar = ()
 
for indice_fila in range(len(nombres_idiomas)):
    # Empezamos en indice_fila + 1: la matriz es simétrica, evitamos repetir.
    for indice_columna in range(indice_fila + 1, len(nombres_idiomas)):
        valor = matriz_similitud[indice_fila][indice_columna]
        if valor > mayor_similitud:
            mayor_similitud = valor
            par_mas_similar = (
                nombres_idiomas[indice_fila], nombres_idiomas[indice_columna]
            )
        if valor < menor_similitud:
            menor_similitud = valor
            par_menos_similar = (
                nombres_idiomas[indice_fila], nombres_idiomas[indice_columna]
            )
 
print(f"\nPar más similar:  {par_mas_similar} -> {mayor_similitud:.2f}")
print(f"Par menos similar: {par_menos_similar} -> {menor_similitud:.2f}")

### [Ex-L1-C8-03] — Mini-sistema de recomendación por palabras clave

In [ ]:
"""Mini-recomendador de contenido por coincidencia de palabras clave."""
 
TOP_N_DEFECTO = 3
 
 
def calcular_relevancia(tags_articulo, palabras_consulta):
    """Calcula la relevancia como tamaño de la intersección entre tags y
    consulta."""
    # La intersección da las palabras comunes; len() las cuenta.
    return len(tags_articulo & palabras_consulta)
 
 
def recomendar(consulta, catalogo, top_n=TOP_N_DEFECTO):
    """Recomienda los artículos más relevantes para una consulta de texto.
 
    Args:
        consulta (str): palabras clave del usuario, separadas por espacios.
        catalogo (list): lista de dicts con claves 'titulo' y 'tags'.
        top_n (int): número de recomendaciones a devolver.
 
    Returns:
        list: lista de tuplas (titulo, relevancia) ordenadas, o lista vacía.
    """
    # --- Validación defensiva de las entradas ---
    try:
        # Una consulta vacía o que no es texto no puede procesarse.
        if not isinstance(consulta, str) or len(consulta.strip()) == 0:
            raise ValueError("la consulta está vacía o no es texto")
        # top_n debe ser un entero positivo.
        if not isinstance(top_n, int) or top_n <= 0:
            raise ValueError("top_n debe ser un entero positivo")
    except ValueError as error:
        print(f"Error en la consulta: {error}")
        return []
 
    # Convertimos la consulta en un set de palabras en minúsculas.
    palabras_consulta = set(consulta.lower().split())
 
    # --- Calculamos la relevancia de cada artículo ---
    articulos_puntuados = []
    for articulo in catalogo:
        try:
            # Acceso a las claves esperadas; si falta alguna -> KeyError.
            tags = articulo["tags"]
            titulo = articulo["titulo"]
        except KeyError as clave_faltante:
            # Captura ESPECÍFICA: un artículo mal formado no tumba el sistema.
            print(f"Advertencia: artículo sin la clave "
                  f"{clave_faltante}, se omite.")
            continue  # saltamos este artículo y seguimos
 
        relevancia = calcular_relevancia(tags, palabras_consulta)
        # Solo nos interesan los artículos con al menos una coincidencia.
        if relevancia > 0:
            articulos_puntuados.append((titulo, relevancia))
 
    # --- Ordenamos por relevancia descendente y recortamos al top_n ---
    # key=lambda par: par[1] ordena por el segundo elemento (la relevancia).
    articulos_puntuados.sort(key=lambda par: par[1], reverse=True)
    return articulos_puntuados[:top_n]
 
 
# --- Catálogo: lista de diccionarios (orientado a registros) ---
catalogo = [
    {"titulo": "Introducción a Python",
     "tags": {"python", "programacion", "basico"}},
    {"titulo": "Pandas para análisis",
     "tags": {"python", "pandas", "datos", "analisis"}},
    {"titulo": "Visualización con Seaborn",
     "tags": {"python", "visualizacion", "datos"}},
    {"titulo": "Machine Learning básico",
     "tags": {"ml", "datos", "modelos", "python"}},
    {"titulo": "Cocina mediterránea",
     "tags": {"cocina", "recetas", "salud"}},
    {"titulo": "SQL desde cero",
     "tags": {"sql", "datos", "bases_de_datos"}},
]
 
# --- Prueba 1: consulta sobre datos en Python ---
print("--- Consulta: 'python datos' ---")
recomendaciones = recomendar("python datos", catalogo, top_n=3)
for titulo, relevancia in recomendaciones:
    print(f"  {titulo} (relevancia: {relevancia})")
 
# --- Prueba 2: consulta sobre cocina ---
print("\n--- Consulta: 'recetas salud' ---")
for titulo, relevancia in recomendar("recetas salud", catalogo):
    print(f"  {titulo} (relevancia: {relevancia})")
 
# --- Prueba 3: consulta inválida (vacía) ---
print("\n--- Consulta vacía ---")
print(recomendar("", catalogo))